Hora de transformar o dado bruto (Bronze) em informação limpa, tipada e estruturada para o negócio, na camada silver da arquitetura medalhão

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, upper, row_number, when, datediff, last, to_date, min, max

# Definição do catálogo e criação do banco de dados "silver"
spark.sql("USE CATALOG medalhao")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [0]:
# Lâ da Bronze
df_customers = spark.table("bronze.tb_customers")


# Deduplicação Sênior com Window Function
window_spec = Window.partitionBy("customer_id").orderBy(col("timestamp_ingestion").desc())

# Cria uma coluna temporária row_num e numera usando window_spec
df_customers_clean = (
    df_customers.withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1) # O registro mais recente será 1
    .drop("row_num") # Remove a coluna temporária
)

# Mapeamento e Transformação
dim_consumidores = df_customers_clean.select(
    col("customer_id").alias("id_consumidor"),
    col("customer_zip_code_prefix").alias("prefixo_cep"),
    col("customer_name").alias("nome_consumidor"),
    upper(col("customer_city")).alias("cidade"),
    upper(col("customer_state")).alias("estado")
)

# Salva
dim_consumidores.write.format("delta").mode("overwrite").saveAsTable("silver.dim_consumidores")

In [0]:
df_orders = spark.table("bronze.tb_orders")

fat_pedidos = ( # Converte de inglês para português
    df_orders
    .withColumn("status",
        when(col("order_status") == "delivered", "entregue")
        .when(col("order_status") == "canceled", "cancelado")
        .when(col("order_status") == "shipped", "enviado")
        .when(col("order_status") == "processing", "processando")
        .when(col("order_status") == "invoiced", "faturado")
        .when(col("order_status") == "unavailable", "indisponível")
        .when(col("order_status") == "created", "criado")
        .when(col("order_status") == "approved", "aprovado")

)
# Calcula o tempo de entrega real
.withColumn("tempo_entrega_dias", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))

# Calcula o tempo estimado de entrega
.withColumn("tempo_entrega_estimado_dias", datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp")))

# Diferença entre o tempo estimado e o tempo real
.withColumn("diferenca_entrega_dias", col("tempo_entrega_dias") - col("tempo_entrega_estimado_dias"))

# Indicador textual
.withColumn("entrega_no_prazo",
    when(col("status") != "entregue", "Não Entregue")
    .when(col("diferenca_entrega_dias") <= 0, "Sim")
    .otherwise("Não")
)

.select(
        col("order_id").alias("id_pedido"),
        col("customer_id").alias("id_consumidor"),
        col("status"),
        col("order_purchase_timestamp").alias("data_pedido"),
        col("order_approved_at").alias("data_aprovacao"),
        col("order_delivered_carrier_date").alias("data_envio_transportadora"),
        col("order_delivered_customer_date").alias("data_entrega_cliente"),
        col("order_estimated_delivery_date").alias("data_estimada_entrega"),
        col("tempo_entrega_dias"),
        col("tempo_entrega_estimado_dias"),
        col("diferenca_entrega_dias"),
        col("entrega_no_prazo")
    )
)
fat_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pedidos")

In [0]:
# Adiantar a 9. (Aqui ainda não precisa arredondar)

df_dolar = spark.table("bronze.tb_cotacao_dolar")

# Extrai a data da API
df_dolar = df_dolar.withColumn("data_cotacao_api", to_date(col("dataHoraCotacao")))

# Início e fim dos dados da API
datas_limites = df_dolar.select(min("data_cotacao_api").alias("inicio"), max("data_cotacao_api").alias("fim")).first()
data_inicio = datas_limites["inicio"]
data_fim = datas_limites["fim"]

# Cria o DataFrame com o calendário contínuo
df_calendario = spark.sql(f"SELECT explode(sequence(to_date('{data_inicio}'), to_date('{data_fim}'), interval 1 day)) as data_calendario")

# Join do calendário completo com os dados da API. Sábados e domingos ficarão nulos nesta etapa
df_join = df_calendario.join(df_dolar, df_calendario.data_calendario == df_dolar.data_cotacao_api, "left")

# Preenche o valor da sexta-feira para o sábado e domingo
window_dolar = Window.orderBy("data_calendario").rowsBetween(Window.unboundedPreceding, Window.currentRow)

dim_cotacao_dolar = df_join.withColumn(
    "cotacao_compra", 
    last("cotacaoCompra", ignorenulls=True).over(window_dolar)
).select(
    col("data_calendario").alias("data_cotacao"),
    col("cotacao_compra")
)

dim_cotacao_dolar.write.format("delta").mode("overwrite").saveAsTable("silver.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# 3. silver.fat_itens_pedidos
df_order_items = spark.table("bronze.tb_order_items")

fat_itens_pedidos = df_order_items.select(
    col("order_id").alias("id_pedido"),
    col("order_item_id").alias("id_item"),
    col("product_id").alias("id_produto"),
    col("seller_id").alias("id_vendedor"),
    col("price").alias("preco_BRL"),
    col("freight_value").alias("preco_frete")
)

# Salva
fat_itens_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_itens_pedidos")
print("Tabela silver.fat_itens_pedidos criada com sucesso!")

# 4. silver.fat_pagamentos_pedidos (respeitando a formatação)
df_order_payments = spark.table("bronze.tb_order_payments")

fat_pagamentos_pedidos = (
    df_order_payments
    .withColumn("tipo_pagamento", 
        when(col("payment_type") == "credit_card", "Cartão de Crédito")
        .when(col("payment_type") == "boleto", "Boleto")
        .when(col("payment_type") == "voucher", "Voucher")
        .when(col("payment_type") == "debit_card", "Cartão de Débito")
        .when(col("payment_type") == "not_defined", "Não Definido")
        .otherwise(col("payment_type"))
    )
    .select(
        col("order_id").alias("id_pedido"),
        col("payment_sequential").alias("sequencial_pagamento"),
        col("tipo_pagamento"),
        col("payment_installments").alias("parcelas_pagamento"),
        col("payment_value").alias("valor_pagamento")
    )
)

fat_pagamentos_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pagamentos_pedidos")
print("Tabela silver.fat_pagamentos_pedidos criada com sucesso!")

Tabela silver.fat_itens_pedidos criada com sucesso!
Tabela silver.fat_pagamentos_pedidos criada com sucesso!


In [0]:
from pyspark.sql.functions import try_to_timestamp, current_timestamp, coalesce, lit

# 5. silver.fat_avaliacoes_pedidos
df_reviews = spark.table("bronze.tb_order_reviews")

fat_avaliacoes_pedidos = (
    df_reviews
    # try_to_timestamp nas datas de texto
    .withColumn("data_criacao_segura", try_to_timestamp(col("review_creation_date")))
    .withColumn("data_resposta_segura", try_to_timestamp(col("review_answer_timestamp")))
    
    # Remover linhas com datas inválidas
    .filter(col("order_id").isNotNull())
    .filter(col("data_criacao_segura") <= current_timestamp())
    
    # Preenchimento de valores vazios
    .withColumn("titulo_tratado", coalesce(col("review_comment_title"), lit("Sem título")))
    .withColumn("comentario_tratado", coalesce(col("review_comment_message"), lit("Sem comentário")))
    
    # Mapeamento
    .select(
        col("review_id").alias("id_avaliacao"),
        col("order_id").alias("id_pedido"),
        col("review_score").alias("nota_avaliacao"),
        col("titulo_tratado").alias("titulo_avaliacao_comentario"),
        col("comentario_tratado").alias("mensagem_avaliacao_comentario"),
        col("data_criacao_segura").alias("data_criacao_avaliacao"),
        col("data_resposta_segura").alias("data_resposta_avaliacao")
    )
)
fat_avaliacoes_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_avaliacoes_pedidos")
print("5. Tabela silver.fat_avaliacoes_pedidos criada!")

# 6. silver.dim_produtos
df_products = spark.table("bronze.tb_products")
window_produtos = Window.partitionBy("product_id").orderBy(col("timestamp_ingestion").desc())

dim_produtos = (
    df_products
    .withColumn("row_num", row_number().over(window_produtos))
    .filter(col("row_num") == 1) # Deduplicação Sênior
    .select( # Mapar
        col("product_id").alias("id_produto"),
        col("product_name").alias("nome_produto"),
        col("product_category_name").alias("categoria_produto"),

        col("product_weight_g").alias("peso_produto_gramas"),
        col("product_length_cm").alias("comprimento_centimetros"),
        col("product_height_cm").alias("altura_centimetros"),
        col("product_width_cm").alias("largura_centimetros"),
        col("product_photos_qty").alias("quantidade_fotos"),
        col("product_name_lenght").alias("tamanho_nome_produto"),
        col("product_description_lenght").alias("tamanho_descricao_produto"),

    )
)
dim_produtos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_produtos")
print("6. Tabela silver.dim_produtos criada!")

# 7. silver.dim_vendedores
df_sellers = spark.table("bronze.tb_sellers")
window_vendedores = Window.partitionBy("seller_id").orderBy(col("timestamp_ingestion").desc())

dim_vendedores = (
    df_sellers
    .withColumn("row_num", row_number().over(window_vendedores))
    .filter(col("row_num") == 1) # Deduplicação Sênior
    .select(
        col("seller_id").alias("id_vendedor"),
        col("seller_name").alias(("nome_vendedor")),
        col("seller_zip_code_prefix").alias("prefixo_cep"),
        upper(col("seller_city")).alias("cidade"), # Cidades em maiúsculo
        upper(col("seller_state")).alias("estado") # Estados em maiúsculo
    )
)
dim_vendedores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_vendedores")
print("7. Tabela silver.dim_vendedores criada!")

# 8. silver.dim_categoria_produtos_traducao
df_traducao = spark.table("bronze.tb_product_category_name_translation")

dim_categoria_produtos_traducao = df_traducao.select(
    col("product_category_name").alias("nome_produto_pt"),
    col("product_category_name_english").alias("nome_produto_en")
)
dim_categoria_produtos_traducao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_categoria_produtos_traducao")
print("8. Tabela silver.dim_categoria_produtos_traducao criada!")

5. Tabela silver.fat_avaliacoes_pedidos criada!
6. Tabela silver.dim_produtos criada!
7. Tabela silver.dim_vendedores criada!
8. Tabela silver.dim_categoria_produtos_traducao criada!


In [0]:
from pyspark.sql.functions import format_number
# Para formatar número do tipo x,y0 para duas casas

# Carregar as tabelas
df_pedidos = spark.table("silver.fat_pedidos")
df_pagamentos = spark.table("silver.fat_pagamentos_pedidos")
df_dolar = spark.table("silver.dim_cotacao_dolar")

# Agrupar pagamentos por pedido
df_pagamentos_agg = (
    df_pagamentos
    .groupBy("id_pedido")
    .sum("valor_pagamento")
    .withColumnRenamed("sum(valor_pagamento)", "valor_total_pago_brl")
)

# Unir Pedidos com Pagamentos
df_final = df_pedidos.join(df_pagamentos_agg, "id_pedido", "left")

# Unir com a Cotação do Dólar
df_final = df_final.withColumn("data_referencia", to_date(col("data_pedido")))

df_final = (
    df_final
    .join(df_dolar, df_final.data_referencia == df_dolar.data_cotacao, "left")
)

# Calcular valor em USD e arredondar
fat_pedido_total = (
    df_final
    .withColumn("valor_total_pago_usd", format_number(col("valor_total_pago_brl") / col("cotacao_compra"), 2))

    .withColumn("valor_total_pago_brl", format_number(col("valor_total_pago_brl"), 2))
    
    .select(
        col("id_pedido"),
        col("id_consumidor"),
        col("status"),
        col("valor_total_pago_brl"),
        col("valor_total_pago_usd"),
        col("data_pedido"),
    )
)

# Salvar a tabela final
fat_pedido_total.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pedido_total")

# Otimização
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

print("Tabela silver.fat_pedido_total criada e otimizada com sucesso!")

Tabela silver.fat_pedido_total criada e otimizada com sucesso!


In [0]:
# Tabelas criadas no schema silver até o momento
print("Tabelas no schema Silver")
display(spark.sql("SHOW TABLES IN silver"))

# 1. Head dim_consumidores
print("\nAmostra: dim_consumidores")
display(spark.table("silver.dim_consumidores").limit(5))

# 2. Head fat_pedidos
print("\nAmostra: fat_pedidos")
display(spark.table("silver.fat_pedidos").limit(5))

# 3. Head fat_itens_pedidos
print("\nAmostra: fat_itens_pedidos")
display(spark.table("silver.fat_itens_pedidos").limit(5))

# 4. Head fat_pagamentos_pedidos
print("\nAmostra: fat_pagamentos_pedidos")
display(spark.table("silver.fat_pagamentos_pedidos").limit(5))

# 5. Head fat_avaliacoes_pedidos
print("\nAmostra: fat_avaliacoes_pedidos")
display(spark.table("silver.fat_avaliacoes_pedidos").limit(5))

# 6. Head dim_produtos
print("\nAmostra: dim_produtos")
display(spark.table("silver.dim_produtos").limit(5))

# 7. Head dim_vendedores
print("\nAmostra: dim_vendedores")
display(spark.table("silver.dim_vendedores").limit(5))

# 8. Head dim_categoria_produtos_traducao
print("\nAmostra: dim_categoria_produtos_traducao")
display(spark.table("silver.dim_categoria_produtos_traducao").limit(5))

# 9. Calendário contínuo e o preenchimento de furos no dólar
print("\nAmostra: dim_cotacao_dolar")
# Ordena pela data
display(spark.table("silver.dim_cotacao_dolar").orderBy("data_cotacao").limit(10))

# 10. Head fat_pedido_total 10 linhas
print("\nAmostra: fat_pedido_total")
display(spark.table("silver.fat_pedido_total").limit(10))

Tabelas no schema Silver


database,tableName,isTemporary
silver,dim_categoria_produtos_traducao,false
silver,dim_consumidores,false
silver,dim_cotacao_dolar,false
silver,dim_produtos,false
silver,dim_vendedores,false
silver,fat_avaliacoes_pedidos,false
silver,fat_itens_pedidos,false
silver,fat_pagamentos_pedidos,false
silver,fat_pedido_total,false
silver,fat_pedidos,false



Amostra: dim_consumidores


id_consumidor,prefixo_cep,nome_consumidor,cidade,estado
00012a2ce6f8dcda20d059ce98491703,6273,Dr. Davi Pinto,OSASCO,SP
000161a058600d5901f007fab4c27140,35550,Sr. Ravi Lucca Sousa,ITAPECERICA,MG
0001fd6190edaaf884bcaf3d49edf079,29830,Giovanna Ramos,NOVA VENECIA,ES
0002414f95344307404f0ace7a26f1d5,39664,Lívia Nogueira,MENDONCA,MG
000379cdec625522490c315e70c7a9fb,4841,Srta. Maria Laura Moura,SAO PAULO,SP



Amostra: fat_pedidos


...

nome_produto_pt,nome_produto_en
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor



Amostra: dim_cotacao_dolar


data_cotacao,cotacao_compra
2016-01-04,4.038
2016-01-05,4.0108
2016-01-06,4.0297
2016-01-07,4.0469
2016-01-08,4.0244
2016-01-09,4.0244
2016-01-10,4.0244
2016-01-11,4.0147
2016-01-12,4.0293
2016-01-13,3.9857



Amostra: fat_pedido_total


id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,38.71,12.24,2017-10-02T10:56:33.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,141.46,37.77,2018-07-24T20:41:37.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,179.12,47.75,2018-08-08T08:38:49.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,72.20,22.02,2017-11-18T19:28:06.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,28.62,8.72,2018-02-13T21:18:39.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,entregue,175.26,53.29,2017-07-09T21:57:05.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,faturado,65.95,20.99,2017-04-11T12:22:08.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,entregue,75.16,24.31,2017-05-16T13:10:30.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,entregue,35.95,11.38,2017-01-23T18:29:09.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,entregue,169.76,53.98,2017-07-29T11:55:02.000Z
